In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("HW_2a_classes_and_simulation.ipynb")

# Homework 2a: Simulating a half car

Slides for the homework: https://docs.google.com/presentation/d/1y6dZiGuSJBfha_5wPd5DEESuZWS8VUTdkc9zYLLOIoI/edit?usp=sharing

In the labs, you learned how to draw, animate, and simulate a quarter car as it travels down a road. You learned about state space models, and how computer simulations operate over discrete steps with a specific time delta, `dt`, and how the choice of `dt` is important for making sure your simulation is accurate without being too slow.

In this homework (part a), you will combine two quarter cars to form a _half car_, draw it, and simulate it driving down a road. You will do that work in **`car.py`**, continuing the `QuarterCar` class from Labs 5 and 6. In part b you will experiment with different half car parameters and roads.

Refer to the assignment slides for a diagram of a half car.

While the quarter car state space model solved for $y_u$ and $y_s$, the half car system has four variables we need to solve for:

* $y_s$: y position of the sprung mass (body), which is now attached to a front and back suspension
* $\theta$: angle of the sprung mass, since the car body will tilt as it moves
* $y_{uf}$: y position of the front unsprung mass (front wheel)
* $y_{ub}$: y position of the back unsprung mass (back wheel)

And it now has two input variables:

* $y_{rf}$: y position of the road under the front tire.
* $y_{rb}$: y position of the road under the back tire.

Both of those input variables can be derived from the same road, but at different `x` offsets.

The system shares the same "fixed" properties of the quarter car, but they are doubled as we now have a front and back of a car:

* $k_{sf}$, $k_{sb}$: Spring constant for the front and back suspensions' spring.
* $c_{sf}$, $c_{sb}$: Damping constant of front and back suspensions' damper.
* $k_{tf}$, $k_{tb}$: Spring constant for front and back tires.

The system also has two new fixed properties related to the shared sprung mass:

* $a_f$, $a_b$: Distance between center of gravity of the car and the front and back of the car, respectively.

### Working in `car.py`

Most of the code for this homework goes in **`car.py`** (in this folder), not in this notebook. Search that file for `# TODO HW 2a` to find every spot you need to edit; each TODO has `# GUIDES` comments underneath it with the details. The import cell below turns on `autoreload`, so the notebook picks up your edits to `car.py` when you re-run a cell. If something looks stale, re-run the import cell (or Restart and Run All). You can also run `python car.py` in a terminal to execute the self-checks at the bottom of the file.


In [ ]:
# Imports
import numpy as np
from scipy.stats import gamma
from scipy.signal import StateSpace, lsim
import matplotlib.pyplot as plt
import matplotlib.animation as animation
# Enable animations to work
%matplotlib widget

%load_ext autoreload
%autoreload 2
from car import QuarterCar, HalfCar


## Part 1: Drawing a half car

Before we dive into the state space system for a half car, let's define a `HalfCar` class that can draw a half car.

Refer to the homework slides for what the half car should look like.

The half car consists of:

* A quarter car in the back, drawn the same way as `QuarterCar`, except with the tire spring drawn right under the suspension spring.
* A quarter car in the front, which is drawn in a similar way to the back quarter car, but with the damper situated _behind_ the spring rather than in front.
* A single sprung mass shared between the quarter cars.

We should be able to re-use much of our `QuarterCar` logic to draw this new car.

One interesting difference between the `QuarterCar` and the `HalfCar` is that the length of the sprung mass (car body), which visually impacts how we draw the car, is a factor in the simulation as well ($a_f$ and $a_b$). We will pass these properties in as arguments to the `HalfCar`'s `draw` function.

**Edit `HalfCar` and `QuarterCar.draw` in `car.py`** (look for `# TODO HW 2a`; the `# GUIDES HW 2a` comments under each TODO walk through the changes step by step). Make sure you already copied your Lab 5 `draw_*` functions into `car.py`.



In [ ]:
# GUIDES HW 2a: Your draw_* functions and QuarterCar should already be in car.py from Lab 5.
# Follow the # TODO HW 2a comments in car.py to:
#   1. Extend QuarterCar.draw for half-car mode (w_s == 0, and d_s < 0 for the front quarter car).
#   2. Implement HalfCar.__init__ (create self.front and self.back).
#   3. Implement HalfCar.draw (draw the sprung mass with axs.fill, since it may be at an angle).
# Re-run this cell after editing car.py.
from car import QuarterCar, HalfCar

In [ ]:
# Let's draw a half car!
half_car = HalfCar()

fig_half_car, axs_half_car = plt.subplots()
half_car.draw(axs=axs_half_car, x=10, y_rf=0, y_uf=0, y_rb=0, y_ub=0, y_s=0, a_f=1.4, a_b=1.4, theta=0)
plt.show()

In [ ]:
grader.check("draw_half_car")

## Part 2: Simulating the half car with a state space model

It's a bit more work, and requires one new mathematical trick, but we can also distill a half car into a state space system.

First, I will describe the state space system. Then, we will put it to work to simulate the half car.

### $y_s$: the y position of the sprung mass (body)

The sprung mass has two springs and two dampers acting on it, but the force is tempered by the angle of the sprung mass ($\theta$):

$$m\ddot{y_s} = -k_{sf}(y_s - y_{uf} - a_{f}\sin{\theta}) - k_{sb}(y_s - y_{ub} + a_{b}\sin{\theta}) - c_{f}(\dot{y_s} - \dot{y_{uf}} - a_{f}\dot{\theta}\cos{\theta}) - c_b(\dot{y_s} - \dot{y_ub} + a_{b}\dot{\theta}\cos{\theta})$$

We can simplify this expression with the [small-angle approximation](https://en.wikipedia.org/wiki/Small-angle_approximation), which states that, for small angles:

$$\sin{\theta} \approx \theta$$

$$\cos{\theta} \approx 1$$

We expect that the body is pitched by only a small angle, so we will trade simulation fidelity for a simpler system by applying this substitution, giving us:

$$m_s\ddot{y_s} = -k_{sf}(y_s - y_{uf} - a_{f}\theta) - k_{sb}(y_s - y_{ub} + a_{b}\theta) - c_{f}(\dot{y_s} - \dot{y_{uf}} - a_{f}\dot{\theta}) - c_b(\dot{y_s} - \dot{y_{ub}} + a_{b}\dot{\theta})$$

Like before, we can move terms around and solve for $\ddot{y_s}$ in terms of our four state variables:

$$m_s\ddot{y_s} = -k_{sf}y_s + k_{sf}y_{uf} + k_{sf}a_{f}\theta - k_{sb}y_s + k_{sb}y_{ub} - k_{sb}a_{b}\theta - c_{f}\dot{y_s} + c_{f}\dot{y_{uf}} + c_{f}a_{f}\dot{\theta} - c_b\dot{y_s} + c_b\dot{y_{ub}} - c_{b}a_{b}\dot{\theta}$$

$$m_s\ddot{y_s} = y_s(-k_{sf}-k_{sb}) + k_{sf}y_{uf} + \theta(k_{sf}a_{f} - k_{sb}a_{b}) + k_{sb}y_{ub} + c_{f}\dot{y_{uf}} + (c_{f}a_{f} - c_ba_{b})\dot{\theta} - \dot{y_s}(c_b + c_{f}) + c_b\dot{y_{ub}}$$

$$\ddot{y_s} = \frac{-k_{sf}-k_{sb}}{m_s}y_s + \frac{k_{sf}}{m_s}y_{uf} + \frac{k_{sb}}{m_s}y_{ub} + \frac{k_{sf}a_{f} - k_{sb}a_{b}}{m_s}\theta + \frac{-c_b - c_{f}}{m_s}\dot{y_s} + \frac{c_{f}}{m_s}\dot{y_{uf}} + \frac{c_b}{m_s}\dot{y_{ub}} + \frac{c_{f}a_{f} - c_ba_{b}}{m_s}\dot{\theta}$$

### $y_{uf}$ and $y_{ub}$: y position of the front and back unsprung masses (wheels)

Both of these positions are governed by the same equations:

$$m_{uf}\ddot{y_{uf}} = k_f(y_s - y_{uf} - a_f\sin{\theta}) + c_f(\dot{y_s} - \dot{y_{uf}} - a_{f}\dot{\theta}\cos{\theta}) - k_{tf}(y_{uf} - y_{rf})$$

$$m_{ub}\ddot{y_{ub}} = k_b(y_s - y_{ub} + a_b\sin{\theta}) + c_b(\dot{y_s} - \dot{y_{ub}} + a_b\dot{\theta}\cos{\theta}) - k_{tb}(y_{ub} - y_{rb})$$

Once again, we can apply the small angle approximation to simplify:

$$m_{uf}\ddot{y_{uf}} = k_f(y_s - y_{uf} - a_f\theta) + c_f(\dot{y_s} - \dot{y_{uf}} - a_{f}\dot{\theta}) - k_{tf}(y_{uf} - y_{rf})$$

$$m_{ub}\ddot{y_{ub}} = k_b(y_s - y_{ub} + a_b\theta) + c_b(\dot{y_s} - \dot{y_{ub}} + a_b\dot{\theta}) - k_{tb}(y_{ub} - y_{rb})$$

Multiply out, and we get:

$$\ddot{y_{uf}} = \frac{k_f}{m_{uf}}y_s + \frac{-k_f-k_tf}{m_{uf}}y_{uf} + \frac{-a_fk_f}{m_{uf}}\theta + \frac{c_f}{m_{uf}}\dot{y_s} + \frac{-c_f}{m_{uf}}\dot{y_{uf}} + \frac{-a_{f}c_{f}}{m_{uf}}\dot{\theta} + \frac{k_{tf}}{m_{uf}}y_{rf}$$

$$\ddot{y_{ub}} = \frac{k_b}{m_{ub}}y_s + \frac{-k_b-k_{tb}}{m_{ub}}y_{ub} + \frac{k_{b}a_{b}}{m_{ub}}\theta + \frac{c_b}{m_{ub}}\dot{y_s} + \frac{-c_b}{m_{ub}}\dot{y_{ub}} + \frac{c_{b}a_b}{m_{ub}}\dot{\theta} + \frac{k_{tb}}{m_{ub}}y_{rb}$$

### $\theta$: angle of the sprung mass

$$I_y\ddot{\theta} = a_{f}k_{f}(y_s - y_{uf} - a_f\sin{\theta}) - a_{b}k_{b}(y_s - y_{ub} + a_{b}\sin{\theta}) + a_{f}c_{f}(\dot{y_s} - \dot{y_{uf}}-a_f\dot{\theta}\cos{\theta}) - a_{b}c_{b}(\dot{y_s} - \dot{y_{ub}} + a_b\dot{\theta}\cos{\theta})$$

Once again, small angle approximation makes this simpler:

$$I_y\ddot{\theta} = a_{f}k_{f}(y_s - y_{uf} - a_f\theta) - a_{b}k_{b}(y_s - y_{ub} + a_{b}\theta) + a_{f}c_{f}(\dot{y_s} - \dot{y_{uf}}-a_f\dot{\theta}) - a_{b}c_{b}(\dot{y_s} - \dot{y_{ub}} + a_b\dot{\theta})$$

Multiply out, and we get:

$$\ddot{\theta} = \frac{a_{f}k_{f} - a_{b}k_{b}}{I_y}y_s + \frac{-a_fk_f}{I_y}y_{uf} + \frac{a_bk_b}{I_y}y_{ub} + \frac{-a_{f}^{2}k_f - a_{b}^{2}k_b}{I_y}\theta + \frac{a_fc_f - a_bc_b}{I_y}\dot{y_s} + \frac{-a_fc_f}{I_y}\dot{y_{uf}} + \frac{a_bc_b}{I_y}\dot{y_{ub}} + \frac{a_{f}^{2}c_f - a_b^2c_b}{I_y}\dot{\theta}$$


### State space model

Like before, we can derive the state space model from the equations above.

#### $A$: The transition matrix

| | $y_s$ | $\theta$ | $y_{uf}$ | $y_{ub}$ | $\dot{y_s}$ | $\dot{\theta}$ | $\dot{y_{uf}}$ | $\dot{y_{ub}}$ |
|-|-------|----------|----------|----------|-------------|----------------|----------------|----------------|
| $\dot{y_s}$ | 0 | 0 | 0 | 0 | 1 | 0 | 0| 0|
| $\dot{\theta}$ | 0 | 0 | 0 | 0 | 0 | 1 | 0| 0|
| $\dot{y_{uf}}$ | 0 | 0 | 0 | 0 | 0 | 0 | 1| 0|
| $\dot{y_{ub}}$ | 0 | 0 | 0 | 0 | 0 | 0 | 0| 1|
| $\ddot{y_s}$ | $\frac{-k_{sf}-k_{sb}}{m_s}$ | $\frac{k_{sf}a_{f} - k_{sb}a_{b}}{m_s}$ | $\frac{k_{sf}}{m_s}$ | $\frac{k_{sb}}{m_s}$ | $\frac{-c_b - c_{f}}{m_s}$ | $\frac{c_{f}a_{f} - c_ba_{b}}{m_s}$ | $\frac{c_{f}}{m_s}$ | $\frac{c_b}{m_s}$ |
| $\ddot{\theta}$ | $\frac{a_{f}k_{f} - a_{b}k_{b}}{I_y}$ | $\frac{-a_{f}^{2}k_f - a_{b}^{2}k_b}{I_y}$ | $\frac{-a_fk_f}{I_y}$ | $\frac{a_bk_b}{I_y}$ | $\frac{a_fc_f - a_bc_b}{I_y}$ | $\frac{-a_{f}^{2}c_f - a_b^2c_b}{I_y}$ | $\frac{-a_fc_f}{I_y}$ | $\frac{a_bc_b}{I_y}$ | 
| $\ddot{y_{uf}}$ | $\frac{k_f}{m_{uf}}$ | $\frac{-a_fk_f}{m_{uf}}$ | $\frac{-k_f-k_{tf}}{m_{uf}}$ | 0 | $\frac{c_f}{m_{uf}}$ | $\frac{-a_{f}c_{f}}{m_{uf}}$ | $\frac{-c_f}{m_{uf}}$ | $0$ |
| $\ddot{y_{ub}}$ | $\frac{k_b}{m_{ub}}$ | $\frac{k_{b}a_{b}}{m_{ub}}$ | $0$ | $\frac{-k_b-k_{tb}}{m_{ub}}$ | $\frac{c_b}{m_{ub}}$ | $\frac{c_{b}a_b}{m_{ub}}$ | $0$ | $\frac{-c_b}{m_{ub}}$ |


#### $B$: The input matrix

| | $y_{rf}$ | $y_{rb}$ |
|-|-------|----------|
| $\dot{y_s}$ | 0 | 0 |
| $\dot{\theta}$ | 0 | 0 |
| $\dot{y_{uf}}$ |  0 | 0 |
| $\dot{y_{ub}}$ |  0 | 0 |
| $\ddot{y_s}$ | 0 | 0 |
| $\ddot{\theta}$ | 0 | 0 |
| $\ddot{y_{uf}}$ | $\frac{k_{tf}}{m_{uf}}$ | 0 |
| $\ddot{y_{ub}}$ | 0 | $\frac{k_{tb}}{m_{ub}}$ |

#### $C$: The output matrix

| | $y_s$ | $\theta$ | $y_{uf}$ | $y_{ub}$ | $\dot{y_s}$ | $\dot{\theta}$ | $\dot{y_{uf}}$ | $\dot{y_{ub}}$ |
|-|-------|----------|----------|----------|-------------|----------------|----------------|----------------|
| $y_s$ | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| $\theta$ | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 0 |
| $y_{uf}$  | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0 |
| $y_{ub}$ | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 |

#### $D$: How input influences output

Once again, $D$ is all zeroes.

| | $y_{rf}$ | $y_{rb}$ |
|-|-------|----------|
| $\dot{y_s}$ | 0 | 0 |
| $\dot{\theta}$ | 0 | 0 |
| $\dot{y_{uf}}$  | 0 | 0 |
| $\dot{y_{ub}}$ | 0 | 0 |

### Simulating a half car with the state space system

With the state space system defined, it's time to simulate a half car. While we had you type out all of the matrices in the lab, for this homework, we will provide you with all of the matrices except for `B`.

`HalfCar` uses a `QuarterCar` for the front and a `QuarterCar` for the back. Properties belonging to the front or back of the half car are on `self.front` and `self.back`.

For example, the mass of the back unsprung weight ($m_{ub}$) is `self.back.m_u` in the code.

We also define the mass of the `HalfCar`'s sprung mass to be the combined weight of the sprung mass in the front and back (`m_s()`).

There is no separate output class. `HalfCar.simulate` should store the output variables ($y_s$, $y_{uf}$, $y_{ub}$, $\theta$) and simulation parameters ($x_r$, $y_{rf}$, $y_{rb}$, `velocity`, `time`, `dt`) on the half car itself (`self`). After `simulate()` returns, you can read those arrays from the same object.

In `car.py` (HW 2a TODOs):

* Copy `interpolate_road` from Lab 6 if it is not already there, and add `start_offset=0` and `stop_offset=0`.
* Implement `HalfCar.m_s()`, `HalfCar.B()`, and `HalfCar.simulate()`. `A()`, `C()`, and `D()` are given.
* Leave `self.Iy = 1100` for now (HW 2b will replace that with a better formula).


In [ ]:
# GUIDES HW 2a: Follow the # TODO HW 2a comments in car.py to:
#   1. Copy interpolate_road from Lab 6 into car.py (if it is not already there) and add the
#      start_offset=0 and stop_offset=0 arguments.
#   2. Implement HalfCar.m_s() (sum of the front and back m_s) and HalfCar.B().
#   3. Implement HalfCar.simulate() so that after you call simulate(), the half car stores
#      velocity, time, dt, x_r, y_rf, y_rb, y_s, theta, y_uf, and y_ub on self.
# Leave self.Iy = 1100 for HW 2b. Re-run this cell after editing car.py.
from car import HalfCar

In [ ]:
# Check code: simulate a default half car on a short flat road.
# simulate() stores its results on the half car (and returns it), so half_car_sim.y_s, half_car_sim.time, etc.
# are the simulation output.
half_car_sim = HalfCar()
half_car_sim.simulate(
    velocity=10,
    dt=10,
    x_r=np.array([0.0, 5.0, 11.0]),
    y_r=np.array([0.0, 0.0, 0.0]),
)

In [ ]:
grader.check("simulation_init_function")

<!-- BEGIN QUESTION -->

## Part 3: Animating the half car

Now that `HalfCar` can draw and simulate itself, let's animate it driving down the bump road from Lab 6, the same way you animated the quarter car in that lab.

Paste the road and `animate_car` from Lab 6 into the cells below, then adapt `animate_car` to a half car. The `# GUIDES` comments inside the function list the changes you need to make.

In [ ]:
# GUIDES: Paste code from Lab 6 to make the road X_r, Y_r.


In [ ]:
def animate_car(car, velocity, dt, x_r, y_r, playback_speed=1.0):
    """
    Animate the given half car going down the given road.
    @param car The `HalfCar` to simulate and draw.
    @param velocity The velocity of the car in the x direction in m/s.
    @param dt The time step of the simulation
    @param x_r, y_r The x and y coordinates of the road.
    @param playback_speed A value that is > 0 that indicates the playback speed (e.g., 0.1 is 10x slowdown).
    """
    # GUIDES HW 2a: Paste in animate_car from Lab 6, and adapt it to a half car.
    # You will need to change:
    # 1. `car` is now a HalfCar. Like the quarter car, simulate() stores its results on the car (car.x_r, car.y_s, ...),
    #    but there are more of them (car.theta, car.y_rf, car.y_rb, car.y_uf, car.y_ub).
    # 2. The call to `car.draw`, so that it passes everything HalfCar.draw needs to draw both parts of the car
    #    (y_rf, y_uf, y_rb, y_ub, y_s, a_f, a_b, and theta).
    # 3. The calculation of `x_size_max` to account for the car's length (`a_f` + `a_b`).
    # 4. `car.y_s_static` to use the value from either the front or back quarter car (e.g., `car.front.y_s_static`).
    # 5. The xlim to center the camera on the center of the half car (x_r will contain the x coordinate of the rear tire).
    ...


In [ ]:
# Check code: Animate the half car.
half_car = HalfCar()
half_car_anim = animate_car(half_car, velocity=10, dt=5, x_r=X_r, y_r=Y_r)


<!-- END QUESTION -->

## Hours and collaborators
Required for every assignment - fill out before you hand-in.

Listing names and websites helps you to document who you worked with and what internet help you received in the case of any plagiarism issues. You should list names of anyone (in class or not) who has substantially helped you with an assignment - or anyone you have *helped*. You do not need to list TAs.

Listing hours helps us track if the assignments are too long.

In [ ]:
import os

# List of names (creates a set)
worked_with_names = {"not filled out"}
# List of URLS FA26 (creates a set)
websites = {"not filled out"}
# Approximate number of hours, including lab/in-class time
hours = -1.5

# VS Code stores the path in a special global variable
if '__vsc_ipynb_file__' in globals():
    notebook_path = globals()['__vsc_ipynb_file__']
    notebook_name = os.path.basename(notebook_path)
    json_name = notebook_name[:-6] + "_source.json"
    if not os.path.exists(json_name):
        print(f"Could not find the json file {json_name}; make sure the required VSCode extension is installed and running")


In [ ]:
grader.check("hours_collaborators")

### To submit

Double check your plots.

- Submit this .ipynb file, the .json file, and **`car.py`** through Gradescope, to **HW 2a** (classes and simulation)

Failures: Forgetting to include car.py - or puting it in a folder
